---
## Stage 9: Multimodal Feature Engineering

**วัตถุประสงค์:** สร้าง feature vector สำหรับแต่ละ pair → input ของ ML model

**Input:** `df_clean`, `labeled_pairs` (จาก Stage 8)  
**Output:** `feature_matrix` DataFrame

| Sub-step | Features |
|----------|----------|
| 9.1 | String Similarity (Jaro-Winkler, Levenshtein, Token Sort) |
| 9.2 | TF-IDF Cosine Similarity (bio) ขั้นนี้ไม่ถูก|
| 9.3 | URL & Domain Features |
| 9.4 | Platform & Meta Features |
| 9.5 | Combine All Features |

### SBERT (B2) - Semantic Text Similarity (Optional)
ถ้าต้องการเพิ่ม feature เชิง semantic (SBERT) ให้รัน:
```bash
  pip install rapidfuzz scikit-learn pandas numpy
  pip install deepface open-clip-torch imagehash Pillow requests torch torchvision
  pip install sentence-transformers   # สำหรับ B2 SBERT (optional)
  pip install transformers accelerate # สำหรับ BLIP-2 (optional)
```
แล้วโมเดล (all-mpnet-base-v2) จะดาวน์โหลดอัตโนมัติ (~420MB)
แนะนำให้ใช้ GPU (ถ้ามี) เพื่อให้การ encode เร็วขึ้น

In [5]:
!pip install rapidfuzz scikit-learn pandas numpy
!pip install deepface open-clip-torch imagehash Pillow requests torch torchvision
!pip install sentence-transformers
!pip install transformers accelerate

In [ ]:
"""
Stage 9 (Complete): Feature Engineering — Cross-Platform Identity Resolution
=============================================================================
รวมทุก feature group จากที่ออกแบบไว้ทั้งหมด:

TEXT & ATTRIBUTE
  B1  — String similarity    : Jaro-Winkler, Levenshtein, Token Sort (userName, fullName)
                                TF-IDF cosine (bio) — fit บน train only
  A1  — URL signal           : domain Jaccard, exact match, personal domain overlap
  A2  — Location             : Haversine distance + text fuzzy match
  A3  — Mention signal       : bio_mentions Jaccard + username cross-reference
  A4  — Hashtag signal       : #hashtag Jaccard (extracted from bio)
  B2  — SBERT semantic       : all-mpnet-base-v2 cosine (optional, GPU แนะนำ)
  B3  — Stylometric          : caps ratio, avg word len, bio length ratio, punctuation
  META— Platform encoding    : same_platform flag, platform pair

IMAGE (D1–D6) — Missing-Aware
  D1  — ArcFace (DeepFace)   : face identity 512-d cosine (SOTA 2024-2025)
  D2  — CLIP ViT-L/14        : visual context 768-d cosine + category
  D3  — pHash                : perceptual hash Hamming distance
  D4  — CLIP zero-shot       : has_face, has_pet, has_logo flags
  D5  — Cross-modal          : pic_type_match
  D6  — Availability mask    : modality_mask สำหรับ attention layer

Input:
  nomalized_profiles.csv  (profile_id, platform, userName, fullName, bio,
                            externalUrl_clean, external_domain, location,
                            location_type, latitude, longitude, bio_mentions,
                            bio_mentions_count, pictureURL)
  labeled_pairs.parquet   (profile_id_a, profile_id_b, label, pair_type)

Output:
  feature_matrix_train.parquet
  feature_matrix_val.parquet
  feature_matrix_test.parquet
  tfidf_vectorizer.pkl
  feature_cols.pkl
  image_cache.pkl           (cache รูปที่ download แล้ว)

Install:
  pip install rapidfuzz scikit-learn pandas numpy
  pip install deepface open-clip-torch imagehash Pillow requests torch torchvision
  pip install sentence-transformers   # สำหรับ B2 SBERT (optional)
  pip install transformers accelerate # สำหรับ BLIP-2 (optional)

Data Leakage Rules:
  - TF-IDF: fit บน train bios เท่านั้น → transform val/test
  - StandardScaler: fit บน train เท่านั้น (ทำใน Stage 10)
  - SBERT / CLIP / ArcFace: stateless — precompute ทั้งหมดได้เลย
  - Image cache: download ครั้งเดียว ใช้ได้ทุก split
"""

from __future__ import annotations

import ast
import io
import math
import pickle
import re
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# ─── optional heavy imports (lazy) ───────────────────────────────────────────
_rapidfuzz_jw  = None
_rapidfuzz_lev = None
_rapidfuzz_fuzz = None

def _get_rapidfuzz():
    global _rapidfuzz_jw, _rapidfuzz_lev, _rapidfuzz_fuzz
    if _rapidfuzz_jw is None:
        from rapidfuzz.distance import JaroWinkler, Levenshtein as Lev
        from rapidfuzz import fuzz
        _rapidfuzz_jw  = JaroWinkler
        _rapidfuzz_lev = Lev
        _rapidfuzz_fuzz = fuzz
    return _rapidfuzz_jw, _rapidfuzz_lev, _rapidfuzz_fuzz


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — SHARED UTILITIES
# ═══════════════════════════════════════════════════════════════════════════════

def _safe_str(val) -> str:
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return ""
    return str(val).strip()

def _parse_list_field(val) -> list:
    """'[a, b]' string หรือ plain string → list"""
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return []
    s = str(val).strip()
    if not s or s in ("nan", "NaN", ""):
        return []
    if s.startswith("["):
        try:
            return [str(x).strip() for x in ast.literal_eval(s) if str(x).strip()]
        except Exception:
            pass
    return [s]

def _parse_mentions(val) -> set:
    s = _safe_str(val)
    if not s:
        return set()
    return {p.strip().lower() for p in re.split(r"\s*\|\s*|\s+", s) if p.strip()}

def _extract_hashtags(bio: str) -> set:
    return {t.lower() for t in re.findall(r"#(\w+)", bio)} if bio else set()

def _jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 0.0
    u = a | b
    return len(a & b) / len(u) if u else 0.0

def _haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = (math.sin(math.radians(lat2 - lat1) / 2) ** 2
         + math.cos(p1) * math.cos(p2)
         * math.sin(math.radians(lon2 - lon1) / 2) ** 2)
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — TEXT & ATTRIBUTE FEATURE COMPUTERS
# ═══════════════════════════════════════════════════════════════════════════════

class PairFeatureComputer:
    """Static methods ที่คำนวณ feature dict สำหรับ 1 pair"""

    # ── B1: String Similarity ────────────────────────────────────────────────
    @staticmethod
    def feat_B1_text(row_a, row_b, tfidf_matrix=None) -> dict:
        JW, Lev, fuzz = _get_rapidfuzz()
        feats = {}
        for field in ["userName", "fullName"]:
            a = _safe_str(row_a.get(field, ""))
            b = _safe_str(row_b.get(field, ""))
            max_len = max(len(a), len(b), 1)
            feats[f"{field}_jaro"]       = JW.normalized_similarity(a, b)
            feats[f"{field}_lev"]        = 1.0 - Lev.distance(a, b) / max_len
            feats[f"{field}_token_sort"] = fuzz.token_sort_ratio(a, b) / 100.0
            feats[f"{field}_exact"]      = float(a == b and a != "")
        # TF-IDF cosine (fit-on-train-only)
        if tfidf_matrix is not None:
            try:
                cos = cosine_similarity(
                    tfidf_matrix[row_a.name], tfidf_matrix[row_b.name]
                )[0][0]
                feats["bio_tfidf_cosine"] = float(cos)
            except Exception:
                feats["bio_tfidf_cosine"] = 0.0
        else:
            feats["bio_tfidf_cosine"] = 0.0
        return feats

    # ── A1: URL Signal ───────────────────────────────────────────────────────
    @staticmethod
    def feat_A1_url(row_a, row_b) -> dict:
        GENERIC = {
            "about.me","twitter.com","facebook.com","instagram.com","youtube.com",
            "linkedin.com","plus.google.com","google.com","bit.ly","goo.gl",
            "youtu.be","fb.com","t.co","ow.ly",
        }
        dom_a  = set(_parse_list_field(row_a.get("external_domain", ""))) - GENERIC
        dom_b  = set(_parse_list_field(row_b.get("external_domain", ""))) - GENERIC
        urls_a = set(_parse_list_field(row_a.get("externalUrl_clean", "")))
        urls_b = set(_parse_list_field(row_b.get("externalUrl_clean", "")))
        return {
            "domain_jaccard":     _jaccard(dom_a, dom_b),
            "domain_exact_match": float(bool(dom_a & dom_b)),
            "url_exact_match":    float(bool(urls_a & urls_b)),
            "url_jaccard":        _jaccard(urls_a, urls_b),
            "domain_count_a":     min(len(dom_a), 10) / 10.0,
            "domain_count_b":     min(len(dom_b), 10) / 10.0,
        }

    # ── A2: Location ─────────────────────────────────────────────────────────
    @staticmethod
    def feat_A2_location(row_a, row_b) -> dict:
        JW, _, fuzz = _get_rapidfuzz()
        lt_a = _safe_str(row_a.get("location_type", "unknown"))
        lt_b = _safe_str(row_b.get("location_type", "unknown"))
        la_a = float(row_a.get("latitude",  0.0) or 0.0)
        lo_a = float(row_a.get("longitude", 0.0) or 0.0)
        la_b = float(row_b.get("latitude",  0.0) or 0.0)
        lo_b = float(row_b.get("longitude", 0.0) or 0.0)
        COORD = {"coordinates", "coordinates_dms"}
        if lt_a in COORD and lt_b in COORD and abs(la_a) > 0.001 and abs(la_b) > 0.001:
            dist       = _haversine_km(la_a, lo_a, la_b, lo_b)
            coord_sim  = float(np.exp(-dist / 100.0))
            same_city  = float(dist < 50.0)
        else:
            coord_sim = same_city = 0.0
        loc_a = _safe_str(row_a.get("location", "")).lower()
        loc_b = _safe_str(row_b.get("location", "")).lower()
        if loc_a and loc_b:
            tj = JW.normalized_similarity(loc_a, loc_b)
            tt = fuzz.token_sort_ratio(loc_a, loc_b) / 100.0
            te = float(loc_a == loc_b)
        else:
            tj = tt = te = 0.0
        lv_a = int(row_a.get("location_valid", 0) or 0)
        lv_b = int(row_b.get("location_valid", 0) or 0)
        return {
            "location_coord_sim":  coord_sim,
            "location_same_city":  same_city,
            "location_text_jaro":  tj,
            "location_text_token": tt,
            "location_text_exact": te,
            "both_have_location":  float(lv_a == 1 and lv_b == 1),
        }

    # ── A3: Mention Signal ───────────────────────────────────────────────────
    @staticmethod
    def feat_A3_mention(row_a, row_b) -> dict:
        m_a  = _parse_mentions(row_a.get("bio_mentions", ""))
        m_b  = _parse_mentions(row_b.get("bio_mentions", ""))
        u_a  = _safe_str(row_a.get("userName", "")).lower()
        u_b  = _safe_str(row_b.get("userName", "")).lower()
        c_a  = int(row_a.get("bio_mentions_count", 0) or 0)
        c_b  = int(row_b.get("bio_mentions_count", 0) or 0)
        return {
            "mention_jaccard":           _jaccard(m_a, m_b),
            "mention_exact_overlap":     float(bool(m_a & m_b)),
            "both_have_mentions":        float(c_a > 0 and c_b > 0),
            "username_in_other_mentions": float(
                (u_a != "" and u_a in m_b) or (u_b != "" and u_b in m_a)
            ),
        }

    # ── A4: Hashtag Signal ───────────────────────────────────────────────────
    @staticmethod
    def feat_A4_hashtag(row_a, row_b) -> dict:
        ta = _extract_hashtags(_safe_str(row_a.get("bio", "")))
        tb = _extract_hashtags(_safe_str(row_b.get("bio", "")))
        return {
            "hashtag_jaccard":       _jaccard(ta, tb),
            "hashtag_exact_overlap": float(bool(ta & tb)),
            "both_have_hashtags":    float(bool(ta) and bool(tb)),
            "hashtag_count_a":       min(len(ta), 10) / 10.0,
            "hashtag_count_b":       min(len(tb), 10) / 10.0,
        }

    # ── B2: SBERT Semantic ───────────────────────────────────────────────────
    @staticmethod
    def feat_B2_sbert(row_a, row_b, sbert_embeddings) -> dict:
        if sbert_embeddings is None:
            return {"bio_sbert_cosine": 0.0}
        try:
            e_a = sbert_embeddings[row_a.name]
            e_b = sbert_embeddings[row_b.name]
            return {"bio_sbert_cosine": float(np.dot(e_a, e_b))}  # L2-normalized
        except Exception:
            return {"bio_sbert_cosine": 0.0}

    # ── B3: Stylometric ──────────────────────────────────────────────────────
    @staticmethod
    def feat_B3_stylometric(row_a, row_b) -> dict:
        def _s(bio):
            if not bio:
                return dict(caps=0.0, avgw=0.0, blen=0, pct=0.0)
            lets  = [c for c in bio if c.isalpha()]
            words = bio.split()
            return {
                "caps": sum(c.isupper() for c in lets) / max(len(lets), 1),
                "avgw": float(np.mean([len(w) for w in words])) if words else 0.0,
                "blen": len(bio),
                "pct":  sum(1 for c in bio if c in "!?.,;:-_()[]{}") / max(len(bio), 1),
            }
        sa = _s(_safe_str(row_a.get("bio", "")))
        sb = _s(_safe_str(row_b.get("bio", "")))
        return {
            "style_caps_diff":    abs(sa["caps"] - sb["caps"]),
            "style_avgword_diff": abs(sa["avgw"] - sb["avgw"]) / 10.0,
            "style_biolen_ratio": min(sa["blen"], sb["blen"]) / max(sa["blen"], sb["blen"], 1),
            "style_punct_diff":   abs(sa["pct"]  - sb["pct"]),
        }

    # ── META: Platform ───────────────────────────────────────────────────────
    @staticmethod
    def feat_meta(row_a, row_b) -> dict:
        p_a = _safe_str(row_a.get("platform", "")).lower()
        p_b = _safe_str(row_b.get("platform", "")).lower()
        PAIRS = {
            tuple(sorted(["googleplus", "instagram"])): 0,
            tuple(sorted(["googleplus", "twitter"])):   1,
            tuple(sorted(["instagram",  "twitter"])):   2,
        }
        return {
            "same_platform": float(p_a == p_b and p_a != ""),
            "platform_pair": PAIRS.get(tuple(sorted([p_a, p_b])), 3),
        }


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — IMAGE FEATURE EXTRACTOR (ArcFace + CLIP + pHash + BLIP-2)
# ═══════════════════════════════════════════════════════════════════════════════

# lazy model handles
_deepface_mod = None
_clip_model   = None
_clip_prep    = None
_clip_tok     = None
_blip_proc    = None
_blip_mod     = None
_imagehash_mod = None
_torch_mod    = None

CLIP_CATEGORIES = [
    "a selfie photo of a person",
    "a photo of a pet or animal",
    "a logo or brand icon",
    "a landscape or nature photo",
    "a food or drink photo",
    "an abstract or artistic image",
    "a group photo with multiple people",
]
CATEGORY_LABELS = ["selfie", "pet", "logo", "landscape", "food", "abstract", "group"]

HEADERS  = {"User-Agent": "Mozilla/5.0 (compatible; ResearchBot/1.0)"}
TIMEOUT  = 10
MAX_SIZE = (512, 512)

def _torch():
    global _torch_mod
    if _torch_mod is None:
        import torch
        _torch_mod = torch
    return _torch_mod

def _deepface():
    global _deepface_mod
    if _deepface_mod is None:
        from deepface import DeepFace
        _deepface_mod = DeepFace
    return _deepface_mod

def _clip():
    global _clip_model, _clip_prep, _clip_tok
    if _clip_model is None:
        import open_clip
        t = _torch()
        dev = "cuda" if t.cuda.is_available() else "cpu"
        _clip_model, _, _clip_prep = open_clip.create_model_and_transforms(
            "ViT-L-14", pretrained="openai", device=dev
        )
        _clip_tok = open_clip.get_tokenizer("ViT-L-14")
        _clip_model.eval()
        print(f"[CLIP] ViT-L/14 loaded on {dev}")
    return _clip_model, _clip_prep, _clip_tok

def _imagehash():
    global _imagehash_mod
    if _imagehash_mod is None:
        import imagehash
        _imagehash_mod = imagehash
    return _imagehash_mod

def _blip():
    global _blip_proc, _blip_mod
    if _blip_proc is None:
        from transformers import Blip2Processor, Blip2ForConditionalGeneration
        t = _torch()
        dev = "cuda" if t.cuda.is_available() else "cpu"
        _blip_proc = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
        _blip_mod  = Blip2ForConditionalGeneration.from_pretrained(
            "Salesforce/blip2-opt-2.7b",
            torch_dtype=t.float16 if dev == "cuda" else t.float32,
        ).to(dev)
        _blip_mod.eval()
        print(f"[BLIP-2] Loaded on {dev}")
    return _blip_proc, _blip_mod


def _download_image(url: str) -> Optional["PIL.Image.Image"]:
    import requests
    from PIL import Image
    if not url or not isinstance(url, str) or url.strip() in ("", "nan"):
        return None
    try:
        r = requests.get(url.strip(), headers=HEADERS, timeout=TIMEOUT, stream=True)
        if r.status_code != 200 or "image" not in r.headers.get("Content-Type", ""):
            return None
        img = Image.open(io.BytesIO(r.content)).convert("RGB")
        img.thumbnail(MAX_SIZE)
        return img
    except Exception:
        return None


class ImageFeatureExtractor:
    """
    Extract image features ต่อ profile (precompute) แล้วคำนวณ pair features

    Usage:
        ext = ImageFeatureExtractor(use_blip=False)
        cache = ext.precompute_all(df, cache_path="image_cache.pkl")
        pair_feats = ext.compute_pair_features(cache[pid_a], cache[pid_b])
    """

    def __init__(self, use_blip: bool = False):
        t = _torch()
        self.device   = "cuda" if t.cuda.is_available() else "cpu"
        self.use_blip = use_blip
        self._cat_emb = None  # cached CLIP text embeddings

    # ── precompute 1 profile ─────────────────────────────────────────────────
    def precompute(self, url: str, bio: str = "") -> dict:
        r = dict(url=url, available=False, arcface_emb=None, clip_emb=None,
                 phash=None, clip_cat=None, has_face=False,
                 has_pet=False, has_logo=False, blip_caption=None)
        img = _download_image(url)
        if img is None:
            return r
        r["available"] = True
        r["arcface_emb"], r["has_face"] = self._arcface(img)
        r["clip_emb"],    r["clip_cat"]  = self._clip_embed(img)
        r["has_pet"]  = r["clip_cat"] == "pet"
        r["has_logo"] = r["clip_cat"] == "logo"
        r["phash"]    = self._phash(img)
        if self.use_blip and len(_safe_str(bio)) < 20:
            r["blip_caption"] = self._blip_caption(img)
        return r

    # ── precompute all profiles ───────────────────────────────────────────────
    def precompute_all(self, df: pd.DataFrame,
                        url_col: str = "pictureURL",
                        bio_col: str = "bio",
                        pid_col: str = "profile_id",
                        cache_path: str = "image_cache.pkl") -> dict:
        cache_file = Path(cache_path)
        existing: dict = {}
        if cache_file.exists():
            try:
                existing = pickle.load(open(cache_file, "rb"))
                print(f"[ImageExtractor] Loaded {len(existing)} cached profiles")
            except Exception:
                pass

        result = dict(existing)
        todo   = df[~df[pid_col].astype(str).isin([str(k) for k in existing])].copy()
        n      = len(todo)
        print(f"[ImageExtractor] Processing {n} new profiles ...")

        for step, (_, row) in enumerate(todo.iterrows()):
            if step % 100 == 0:
                print(f"  {step}/{n}", end="\r")
            pid = row.get(pid_col)
            url = _safe_str(row.get(url_col, ""))
            bio = _safe_str(row.get(bio_col, ""))
            cache = self.precompute(url, bio)
            cache.pop("img", None)  # ไม่เก็บ PIL Image ใน cache file
            result[pid] = cache
            if step % 200 == 0 and step > 0:
                pickle.dump(result, open(cache_file, "wb"))

        pickle.dump(result, open(cache_file, "wb"))
        n_ok = sum(1 for v in result.values() if v.get("available"))
        print(f"\n[ImageExtractor] Done. Available: {n_ok}/{len(result)}")
        return result

    # ── pair features ────────────────────────────────────────────────────────
    def compute_pair_features(self, ca: dict, cb: dict) -> dict:
        av_a = ca.get("available", False)
        av_b = cb.get("available", False)
        f = {
            "pic_available_a":    float(av_a),
            "pic_available_b":    float(av_b),
            "pic_available_both": float(av_a and av_b),
            "modality_mask":      float(int(av_a) + int(av_b)),  # 0/1/2
        }
        f.update(self._pair_arcface(ca, cb))
        f.update(self._pair_clip(ca, cb))
        f.update(self._pair_phash(ca, cb))
        # D4 zero-shot flags
        for side, c in [("a", ca), ("b", cb)]:
            f[f"has_face_{side}"] = float(c.get("has_face", False))
            f[f"has_pet_{side}"]  = float(c.get("has_pet",  False))
            f[f"has_logo_{side}"] = float(c.get("has_logo", False))
        # D5 pic type match
        cat_a = ca.get("clip_cat") or ""
        cat_b = cb.get("clip_cat") or ""
        f["pic_type_match"] = float(cat_a == cat_b and cat_a != "" and av_a and av_b)
        return f

    # ── private: ArcFace ─────────────────────────────────────────────────────
    def _arcface(self, img):
        try:
            img_np = np.array(img)
            res = _deepface().represent(
                img_path=img_np, model_name="ArcFace",
                detector_backend="retinaface",
                enforce_detection=False, align=True,
            )
            if res:
                emb = np.array(res[0]["embedding"], dtype=np.float32)
                n   = np.linalg.norm(emb)
                return (emb / n if n > 0 else emb), True
        except Exception:
            pass
        return None, False

    def _pair_arcface(self, ca, cb) -> dict:
        ea, eb = ca.get("arcface_emb"), cb.get("arcface_emb")
        hfa, hfb = ca.get("has_face", False), cb.get("has_face", False)
        cos = float(np.dot(ea, eb)) if ea is not None and eb is not None else 0.0
        cos = max(-1.0, min(1.0, cos))
        return {
            "face_detected_a":    float(hfa),
            "face_detected_b":    float(hfb),
            "both_face_detected": float(hfa and hfb),
            "face_arcface_cosine": cos,
            "face_arcface_match":  float(cos > 0.28),  # ArcFace threshold
        }

    # ── private: CLIP ────────────────────────────────────────────────────────
    def _get_cat_emb(self):
        if self._cat_emb is not None:
            return self._cat_emb
        t = _torch()
        model, _, tok = _clip()
        with t.no_grad():
            tokens   = tok(CLIP_CATEGORIES).to(self.device)
            embs     = model.encode_text(tokens)
            embs     = embs / embs.norm(dim=-1, keepdim=True)
            self._cat_emb = embs.cpu().float().numpy()
        return self._cat_emb

    def _clip_embed(self, img):
        try:
            from PIL import Image as PILImage
            t = _torch()
            model, prep, _ = _clip()
            tensor = prep(img).unsqueeze(0).to(self.device)
            with t.no_grad():
                emb = model.encode_image(tensor)
                emb = emb / emb.norm(dim=-1, keepdim=True)
                emb_np = emb.cpu().float().numpy()[0]
            scores  = emb_np @ self._get_cat_emb().T
            cat     = CATEGORY_LABELS[int(np.argmax(scores))]
            return emb_np, cat
        except Exception:
            return None, None

    def _pair_clip(self, ca, cb) -> dict:
        ea, eb = ca.get("clip_emb"), cb.get("clip_emb")
        if ea is not None and eb is not None:
            cos = float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb) + 1e-8))
            cos = max(-1.0, min(1.0, cos))
        else:
            cos = 0.0
        cat_a = ca.get("clip_cat") or "unknown"
        cat_b = cb.get("clip_cat") or "unknown"
        return {
            "clip_cosine_sim":    cos,
            "clip_category_a":    cat_a,
            "clip_category_b":    cat_b,
            "clip_category_match": float(
                cat_a == cat_b and cat_a != "unknown"
                and ca.get("available") and cb.get("available")
            ),
        }

    # ── private: pHash ───────────────────────────────────────────────────────
    def _phash(self, img):
        try:
            return _imagehash().phash(img)
        except Exception:
            return None

    def _pair_phash(self, ca, cb) -> dict:
        ha, hb = ca.get("phash"), cb.get("phash")
        if ha is not None and hb is not None:
            d = int(ha - hb)
            return {
                "phash_hamming":     d,
                "phash_exact_match": float(d == 0),
                "phash_near_dup":    float(d <= 10),
                "phash_sim":         1.0 - d / 64.0,
            }
        return {"phash_hamming": 64, "phash_exact_match": 0.0,
                "phash_near_dup": 0.0, "phash_sim": 0.0}

    # ── private: BLIP-2 caption ──────────────────────────────────────────────
    def _blip_caption(self, img) -> Optional[str]:
        try:
            t = _torch()
            proc, model = _blip()
            dev  = next(model.parameters()).device
            inp  = proc(img, return_tensors="pt").to(dev, t.float16)
            with t.no_grad():
                out = model.generate(**inp, max_new_tokens=50)
            return proc.decode(out[0], skip_special_tokens=True).strip()
        except Exception:
            return None


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — SBERT HELPER
# ═══════════════════════════════════════════════════════════════════════════════

def compute_sbert_embeddings(df: pd.DataFrame,
                              model_name: str = "all-mpnet-base-v2",
                              batch_size: int = 64) -> np.ndarray:
    """
    Precompute SBERT embeddings สำหรับทุก bio
    Returns np.ndarray shape (n, 768) — L2 normalized
    """
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        raise ImportError("pip install sentence-transformers")
    print(f"[SBERT] Loading {model_name} ...")
    model = SentenceTransformer(model_name)
    bios  = df["bio"].fillna("").tolist()
    print(f"[SBERT] Encoding {len(bios)} bios ...")
    return model.encode(bios, batch_size=batch_size,
                        show_progress_bar=True,
                        normalize_embeddings=True,
                        convert_to_numpy=True)


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — FEATURE ENGINEER (orchestrator)
# ═══════════════════════════════════════════════════════════════════════════════

class FeatureEngineer:
    """
    Orchestrator ที่รวบรวม features ทุก group สำหรับแต่ละ split

    Parameters
    ----------
    df               : nomalized_profiles.csv DataFrame
    feature_groups   : list ของ group IDs ที่ต้องการ
                       ['B1','A1','A2','A3','A4','B2','B3','META','IMG']
    tfidf_bios       : pd.Series ของ bio texts ที่ใช้ fit TF-IDF
                       *** ต้องเป็น train bios เท่านั้น ***
    sbert_embeddings : np.ndarray (n, 768) — precomputed
    image_cache      : dict {profile_id: cache_dict} จาก ImageFeatureExtractor
    use_blip         : เปิด BLIP-2 caption fallback
    """

    def __init__(self,
                 df: pd.DataFrame,
                 feature_groups: Optional[list] = None,
                 tfidf_bios: Optional[pd.Series] = None,
                 sbert_embeddings: Optional[np.ndarray] = None,
                 image_cache: Optional[dict] = None,
                 use_blip: bool = False):

        self.df             = df.copy().reset_index(drop=True)
        self.feature_groups = feature_groups or [
            "B1","A1","A2","A3","A4","B3","META"
        ]
        self.sbert_embeddings = sbert_embeddings
        self.image_cache      = image_cache or {}

        # TF-IDF — fit บน train bios เท่านั้น
        self.tfidf_vectorizer = None
        self.tfidf_matrix     = None
        if "B1" in self.feature_groups and tfidf_bios is not None:
            print("[FE] Fitting TF-IDF on train bios ...")
            self.tfidf_vectorizer = TfidfVectorizer(
                max_features=5000, ngram_range=(1, 2),
                min_df=2, sublinear_tf=True,
            )
            all_bios = self.df["bio"].fillna("").tolist()
            self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(all_bios)
            print(f"   TF-IDF matrix: {self.tfidf_matrix.shape}")

        # Image extractor (lazy — สร้างเมื่อต้องใช้จริง)
        self._img_extractor: Optional[ImageFeatureExtractor] = None
        self._use_blip = use_blip

    def _get_img_extractor(self) -> ImageFeatureExtractor:
        if self._img_extractor is None:
            self._img_extractor = ImageFeatureExtractor(use_blip=self._use_blip)
        return self._img_extractor

    # ── compute pairs ─────────────────────────────────────────────────────────
    def compute_pairs(self, labeled_pairs: pd.DataFrame,
                      split_df: Optional[pd.DataFrame] = None,
                      verbose: bool = True) -> pd.DataFrame:
        """
        คำนวณ feature matrix สำหรับทุก pair

        Parameters
        ----------
        labeled_pairs : DataFrame [profile_id_a, profile_id_b, label, ...]
        split_df      : subset df สำหรับ split นี้ (ถ้า None ใช้ self.df)
        verbose       : แสดง progress
        """
        src = (split_df if split_df is not None else self.df).reset_index(drop=True)

        # TF-IDF transform (transform only — ไม่ refit)
        tfidf_mat = None
        if "B1" in self.feature_groups and self.tfidf_vectorizer is not None:
            tfidf_mat = self.tfidf_vectorizer.transform(src["bio"].fillna("").tolist())

        pid_to_idx = {int(r["profile_id"]): i for i, r in src.iterrows()}

        results = []
        n       = len(labeled_pairs)
        img_ext = self._get_img_extractor() if "IMG" in self.feature_groups else None

        for step, (_, pair) in enumerate(labeled_pairs.iterrows()):
            if verbose and step % 2000 == 0:
                print(f"   {step}/{n} pairs", end="\r")

            pid_a = int(pair["profile_id_a"])
            pid_b = int(pair["profile_id_b"])
            if pid_a not in pid_to_idx or pid_b not in pid_to_idx:
                continue

            row_a = src.loc[pid_to_idx[pid_a]]
            row_b = src.loc[pid_to_idx[pid_b]]

            f = {
                "profile_id_a": pid_a,
                "profile_id_b": pid_b,
                "label":        int(pair.get("label", -1)),
                "pair_type":    str(pair.get("pair_type", "")),
            }

            if "B1"   in self.feature_groups:
                f.update(PairFeatureComputer.feat_B1_text(row_a, row_b, tfidf_mat))
            if "A1"   in self.feature_groups:
                f.update(PairFeatureComputer.feat_A1_url(row_a, row_b))
            if "A2"   in self.feature_groups:
                f.update(PairFeatureComputer.feat_A2_location(row_a, row_b))
            if "A3"   in self.feature_groups:
                f.update(PairFeatureComputer.feat_A3_mention(row_a, row_b))
            if "A4"   in self.feature_groups:
                f.update(PairFeatureComputer.feat_A4_hashtag(row_a, row_b))
            if "B2"   in self.feature_groups:
                f.update(PairFeatureComputer.feat_B2_sbert(
                    row_a, row_b, self.sbert_embeddings))
            if "B3"   in self.feature_groups:
                f.update(PairFeatureComputer.feat_B3_stylometric(row_a, row_b))
            if "META" in self.feature_groups:
                f.update(PairFeatureComputer.feat_meta(row_a, row_b))
            if "IMG"  in self.feature_groups and img_ext is not None:
                ca = self.image_cache.get(pid_a, {"available": False})
                cb = self.image_cache.get(pid_b, {"available": False})
                f.update(img_ext.compute_pair_features(ca, cb))

            results.append(f)

        if verbose:
            print(f"\n[FE] Done — {len(results)} pairs.")

        return pd.DataFrame(results)

    # ── helpers ───────────────────────────────────────────────────────────────
    def get_feature_cols(self, fm: pd.DataFrame) -> list:
        SKIP = {"profile_id_a", "profile_id_b", "label", "pair_type",
                "clip_category_a", "clip_category_b"}
        return [c for c in fm.columns if c not in SKIP]

    def get_feature_summary(self, fm: pd.DataFrame) -> pd.DataFrame:
        rows = []
        for col in self.get_feature_cols(fm):
            if fm[col].dtype not in [np.float64, np.float32, np.int64, np.int32, float, int]:
                continue
            pos = fm[fm["label"] == 1][col]
            neg = fm[fm["label"] == 0][col]
            rows.append({
                "feature":  col,
                "pos_mean": round(pos.mean(), 4),
                "neg_mean": round(neg.mean(), 4),
                "diff":     round(pos.mean() - neg.mean(), 4),
                "null%":    round(fm[col].isna().mean() * 100, 2),
            })
        return pd.DataFrame(rows).sort_values("diff", ascending=False)


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — FILE FINDER UTILITY
# ═══════════════════════════════════════════════════════════════════════════════

def find_data_file(name: str) -> Optional[Path]:
    """ค้นหาไฟล์จาก cwd ขึ้นไปถึง parent 4 ระดับ"""
    cwd = Path.cwd()
    for base in [cwd] + list(cwd.parents)[:4]:
        p = base / name
        if p.exists():
            return p
    for base in [cwd] + list(cwd.parents)[:2]:
        hits = list(base.rglob(name))
        if hits:
            return hits[0]
    return None


# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — MAIN (notebook __main__)
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    print("=" * 60)
    print("Stage 9 (Complete): Feature Engineering")
    print("=" * 60)

    # ── Config ────────────────────────────────────────────────────────────────
    USE_SBERT = False        # True → pip install sentence-transformers
    USE_IMAGE = False        # True → pip install deepface open-clip-torch imagehash
    USE_BLIP  = False        # True → pip install transformers accelerate
    RANDOM_SEED = 42

    # ── 1. Load data ──────────────────────────────────────────────────────────
    profiles_path = (find_data_file("nomalized_profiles.csv")
                     or find_data_file("combined_profiles.csv"))
    if profiles_path is None:
        raise FileNotFoundError("Cannot find nomalized_profiles.csv")

    df = pd.read_csv(profiles_path)
    if "profile_id" not in df.columns:
        df["profile_id"] = range(len(df))
    if "user_folder" in df.columns:
        df["user_folder"] = df["user_folder"].fillna(df["profile_id"].astype(str))

    labeled_pairs = pd.read_parquet("labeled_pairs.parquet")
    print(f"Profiles: {len(df):,}  |  Pairs: {len(labeled_pairs):,}")
    print(f"Label dist: {labeled_pairs['label'].value_counts().to_dict()}")

    # ── 2. Entity-aware 70/15/15 split ───────────────────────────────────────
    unique_pid = df["profile_id"].unique()
    train_pid, tmp_pid = train_test_split(unique_pid, test_size=0.30, random_state=RANDOM_SEED)
    val_pid,  test_pid = train_test_split(tmp_pid,    test_size=0.50, random_state=RANDOM_SEED)

    def _fp(pairs, pid_set):
        m = pairs["profile_id_a"].isin(pid_set) & pairs["profile_id_b"].isin(pid_set)
        return pairs[m].copy()

    train_pairs = _fp(labeled_pairs, set(train_pid))
    val_pairs   = _fp(labeled_pairs, set(val_pid))
    test_pairs  = _fp(labeled_pairs, set(test_pid))

    train_df = df[df["profile_id"].isin(train_pid)].reset_index(drop=True)
    val_df   = df[df["profile_id"].isin(val_pid)].reset_index(drop=True)
    test_df  = df[df["profile_id"].isin(test_pid)].reset_index(drop=True)

    print(f"Split → Train: {len(train_pairs):,}, Val: {len(val_pairs):,}, Test: {len(test_pairs):,}")

    # ── 3. Optional: SBERT embeddings ────────────────────────────────────────
    sbert_emb = None
    if USE_SBERT:
        sbert_emb = compute_sbert_embeddings(df)

    # ── 4. Optional: Image cache ─────────────────────────────────────────────
    img_cache = {}
    if USE_IMAGE:
        ext       = ImageFeatureExtractor(use_blip=USE_BLIP)
        img_cache = ext.precompute_all(df, cache_path="image_cache.pkl")

    # ── 5. Feature groups ─────────────────────────────────────────────────────
    groups = ["B1", "A1", "A2", "A3", "A4", "B3", "META"]
    if USE_SBERT:
        groups.append("B2")
    if USE_IMAGE:
        groups.append("IMG")

    print(f"\nActive feature groups: {groups}")

    # ── 6. Build FeatureEngineer ──────────────────────────────────────────────
    fe = FeatureEngineer(
        df               = df,
        feature_groups   = groups,
        tfidf_bios       = train_df["bio"],   # fit TF-IDF บน train เท่านั้น
        sbert_embeddings = sbert_emb,
        image_cache      = img_cache,
        use_blip         = USE_BLIP,
    )

    # ── 7. Compute feature matrices ───────────────────────────────────────────
    print("\n--- Train ---")
    train_fm = fe.compute_pairs(train_pairs, split_df=train_df)
    print("\n--- Val ---")
    val_fm   = fe.compute_pairs(val_pairs,   split_df=val_df)
    print("\n--- Test ---")
    test_fm  = fe.compute_pairs(test_pairs,  split_df=test_df)

    # ── 8. Summary ────────────────────────────────────────────────────────────
    feat_cols = fe.get_feature_cols(train_fm)
    print(f"\nTotal numeric features: {len(feat_cols)}")
    print("\nTop 20 discriminative features:")
    print(fe.get_feature_summary(train_fm).head(20).to_string(index=False))

    # ── 9. Save ───────────────────────────────────────────────────────────────
    train_fm.to_parquet("feature_matrix_train.parquet", index=False)
    val_fm.to_parquet("feature_matrix_val.parquet",     index=False)
    test_fm.to_parquet("feature_matrix_test.parquet",   index=False)
    pickle.dump(fe.tfidf_vectorizer, open("tfidf_vectorizer.pkl", "wb"))
    pickle.dump(feat_cols,           open("feature_cols.pkl", "wb"))

    print("\n✅ Stage 9 Complete!")
    print(f"   Train: {len(train_fm):,} rows x {len(feat_cols)} features")
    print(f"   Val  : {len(val_fm):,} rows")
    print(f"   Test : {len(test_fm):,} rows")


Stage 9 (Complete): Feature Engineering
Profiles: 36,807  |  Pairs: 24,663,633
Label dist: {0: 20603433, 1: 4060200}
Split → Train: 14,067,793, Val: 459,415, Test: 488,055

Active feature groups: ['B1', 'A1', 'A2', 'A3', 'A4', 'B3', 'META']
[FE] Fitting TF-IDF on train bios ...
   TF-IDF matrix: (36807, 5000)

--- Train ---


In [2]:
python -c "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')"


SyntaxError: invalid syntax (2588376366.py, line 1)

In [ ]:
"""
Stage 8: Pair Construction & Label Building
============================================
สร้าง labeled_pairs.parquet สำหรับ train ML model

3 ประเภท pairs:
  (1) Positive pairs   — profiles ที่ user_folder เดียวกัน คนละ platform → label=1
  (2) Random negatives — profiles ที่ user_folder ต่างกัน สุ่มจับคู่     → label=0
  (3) Hard negatives   — profiles ที่ชื่อ/username คล้ายกันแต่คนละคน    → label=0

Input  : nomalized_profiles.csv  (ที่มี profile_id + platform)
         หรือ combined_profiles.csv (ใช้ user_folder เป็น entity key)
Output : labeled_pairs.parquet
         pair_stats.json  (summary report)

Data Leakage Note:
  - ไม่ใช้ข้อมูลจาก test set ในการ generate pairs
  - entity-aware split จะทำใน Stage 10
  - file นี้ generate pairs ทั้งหมดก่อน แล้วค่อย split ใน Stage 10
"""

import json
import math
import random
import warnings
import itertools
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from rapidfuzz.distance import JaroWinkler

warnings.filterwarnings("ignore")

# ─── config ──────────────────────────────────────────────────────────────────

RANDOM_SEED        = 42
NEG_TO_POS_RATIO   = 5      # random negatives = 5x positive pairs
HARD_NEG_RATIO     = 2      # hard negatives   = 2x positive pairs
HARD_NEG_THRESHOLD = 0.65   # Jaro-Winkler threshold สำหรับ hard negative candidate
CROSS_PLATFORM_ONLY = True  # positive pairs ข้าม platform เท่านั้น (ไม่จับ same platform)

# ─── utilities ───────────────────────────────────────────────────────────────

def _safe_str(val) -> str:
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return ""
    return str(val).strip().lower()


def _name_similarity(a: str, b: str) -> float:
    """Jaro-Winkler similarity ระหว่าง 2 strings"""
    if not a or not b:
        return 0.0
    return JaroWinkler.normalized_similarity(a, b)


# ─── pair generators ─────────────────────────────────────────────────────────

class PairBuilder:
    """
    สร้าง labeled pairs จาก DataFrame ที่มี profile_id + user_folder + platform
    """

    def __init__(self, df: pd.DataFrame, seed: int = RANDOM_SEED):
        """
        Parameters
        ----------
        df   : DataFrame ที่มี columns:
               profile_id (int), user_folder (str), platform (str),
               userName (str), fullName (str)
        seed : random seed สำหรับ reproducibility
        """
        self.df   = df.copy().reset_index(drop=True)
        self.seed = seed
        random.seed(seed)
        np.random.seed(seed)

        # index lookups
        self._build_indexes()

    def _build_indexes(self):
        """สร้าง lookup tables สำหรับ fast access"""

        # ใช้ profile_id เป็น string ทั้งระบบ — ป้องกัน int cast error
        self.df['profile_id'] = self.df['profile_id'].astype(str)

        # entity → list of profile_ids (ใน DataFrame index)
        self.entity_to_rows: dict[str, list[int]] = {}
        for i, row in self.df.iterrows():
            key = str(row['user_folder']) if pd.notna(row.get('user_folder')) else str(row['profile_id'])
            self.entity_to_rows.setdefault(key, []).append(i)

        # profile_id (str) → row index
        self.pid_to_idx: dict[str, int] = {
            str(r['profile_id']): i for i, r in self.df.iterrows()
        }

        # platform → list of row indexes
        self.platform_to_rows: dict[str, list[int]] = {}
        for i, row in self.df.iterrows():
            plat = _safe_str(row.get('platform', ''))
            self.platform_to_rows.setdefault(plat, []).append(i)

        print(f"[PairBuilder] Loaded {len(self.df)} profiles, "
              f"{len(self.entity_to_rows)} entities, "
              f"{len(self.platform_to_rows)} platforms")

    # ──────────────────────────────────────────────────────────
    # (1) POSITIVE PAIRS
    # ──────────────────────────────────────────────────────────

    def generate_positive_pairs(self) -> pd.DataFrame:
        """
        สร้าง positive pairs: profiles ที่ user_folder เดียวกัน คนละ platform
        ใช้ C(k,2) combinations สำหรับ entities ที่อยู่บน k platforms
        """
        pairs = []
        skipped_same_platform = 0

        for entity, row_idxs in self.entity_to_rows.items():
            if len(row_idxs) < 2:
                continue

            # ดึง rows ที่ platform ต่างกัน
            profiles = [self.df.loc[i] for i in row_idxs]

            for r_a, r_b in itertools.combinations(profiles, 2):
                plat_a = _safe_str(r_a.get('platform', ''))
                plat_b = _safe_str(r_b.get('platform', ''))

                # ถ้าเปิด CROSS_PLATFORM_ONLY ให้ข้าม same-platform pairs
                if CROSS_PLATFORM_ONLY and plat_a == plat_b:
                    skipped_same_platform += 1
                    continue

                pairs.append({
                    'profile_id_a': str(r_a['profile_id']),
                    'profile_id_b': str(r_b['profile_id']),
                    'platform_a':   plat_a,
                    'platform_b':   plat_b,
                    'label':        1,
                    'pair_type':    'positive',
                })

        df_pos = pd.DataFrame(pairs)
        print(f"[PairBuilder] Positive pairs: {len(df_pos)}"
              f"  (skipped same-platform: {skipped_same_platform})")
        return df_pos

    # ──────────────────────────────────────────────────────────
    # (2) RANDOM NEGATIVE PAIRS
    # ──────────────────────────────────────────────────────────

    def generate_random_negatives(self, n_positive: int,
                                  ratio: float = NEG_TO_POS_RATIO,
                                  positive_pid_set: set = None) -> pd.DataFrame:
        """
        สุ่ม pairs ที่ entity ต่างกัน

        Parameters
        ----------
        n_positive      : จำนวน positive pairs (เพื่อคำนวณ target size)
        ratio           : จำนวน random negatives = ratio × n_positive
        positive_pid_set: set ของ (pid_a, pid_b) ที่เป็น positive (เพื่อ dedup)
        """
        n_target    = int(n_positive * ratio)
        all_ids     = self.df['profile_id'].astype(str).tolist()

        # lookup: profile_id (str) → user_folder / platform
        pid_to_folder = {
            str(self.df.loc[i, 'profile_id']): str(self.df.loc[i, 'user_folder'])
            for i in range(len(self.df))
            if pd.notna(self.df.loc[i, 'user_folder'])
        }
        pid_to_plat = {
            str(self.df.loc[i, 'profile_id']): _safe_str(self.df.loc[i, 'platform'])
            for i in range(len(self.df))
        }

        positive_set = positive_pid_set or set()
        generated = set()
        pairs = []
        attempts = 0
        max_attempts = n_target * 20

        while len(pairs) < n_target and attempts < max_attempts:
            attempts += 1
            idx_a = random.randint(0, len(all_ids) - 1)
            idx_b = random.randint(0, len(all_ids) - 1)

            pid_a = all_ids[idx_a]
            pid_b = all_ids[idx_b]

            if pid_a == pid_b:
                continue

            # normalize order (string sort)
            key = tuple(sorted([pid_a, pid_b]))
            if key in generated or key in positive_set:
                continue

            # ต้องเป็น entity ต่างกัน (user_folder ต่างกัน)
            folder_a = pid_to_folder.get(pid_a, f"__unk_{pid_a}")
            folder_b = pid_to_folder.get(pid_b, f"__unk_{pid_b}")
            if folder_a == folder_b:
                continue

            # ไม่บังคับ cross-platform — same-platform negative ก็เป็นไปได้ในโลกจริง
            # โมเดลเรียนรู้จาก same_platform feature ใน Stage 9 เอง

            generated.add(key)
            pairs.append({
                'profile_id_a': pid_a,
                'profile_id_b': pid_b,
                'platform_a':   pid_to_plat.get(pid_a, ''),
                'platform_b':   pid_to_plat.get(pid_b, ''),
                'label':        0,
                'pair_type':    'random_negative',
            })

        df_neg = pd.DataFrame(pairs)
        print(f"[PairBuilder] Random negatives: {len(df_neg)}"
              f"  (target={n_target}, attempts={attempts})")
        return df_neg

    # ──────────────────────────────────────────────────────────
    # (3) HARD NEGATIVE PAIRS
    # ──────────────────────────────────────────────────────────

    def generate_hard_negatives(self, n_positive: int,
                                 ratio: float = HARD_NEG_RATIO,
                                 threshold: float = HARD_NEG_THRESHOLD,
                                 positive_pid_set: set = None,
                                 random_neg_set: set = None) -> pd.DataFrame:
        """
        Hard negatives = pairs ที่ username/fullName คล้ายกันมาก
        แต่เป็นคนละ entity (คนละ user_folder)

        Strategy: ใช้ blocking by name_prefix3 แล้ว filter ด้วย Jaro-Winkler
        เพื่อหา pairs ที่ "หน้าตาคล้ายกันแต่คนละคน"
        """
        n_target = int(n_positive * ratio)
        existing = (positive_pid_set or set()) | (random_neg_set or set())

        # สร้าง blocking key: 3 ตัวอักษรแรกของ userName
        self.df['_name_prefix3'] = self.df['userName'].fillna('').str.lower().str[:3]

        # ดึงเฉพาะ profiles ที่มี userName (ไม่ว่าง)
        has_name = self.df[self.df['userName'].notna() &
                           (self.df['userName'].str.strip() != '')].copy()

        # group by prefix
        blocks = has_name.groupby('_name_prefix3', group_keys=False)

        # lookup — ใช้ str ทั้งหมด
        pid_to_folder = {
            str(r['profile_id']): str(r['user_folder'])
            for _, r in self.df.iterrows()
            if pd.notna(r.get('user_folder'))
        }
        pid_to_plat = {
            str(r['profile_id']): _safe_str(r.get('platform', ''))
            for _, r in self.df.iterrows()
        }

        candidates = []
        for prefix, group in blocks:
            if len(prefix) < 2 or len(group) < 2:
                continue

            rows = group.to_dict('records')
            for i in range(len(rows)):
                for j in range(i + 1, len(rows)):
                    ra, rb = rows[i], rows[j]
                    pid_a = str(ra['profile_id'])
                    pid_b = str(rb['profile_id'])

                    # ต้องเป็น entity ต่างกัน
                    folder_a = pid_to_folder.get(pid_a, f"__unk_{pid_a}")
                    folder_b = pid_to_folder.get(pid_b, f"__unk_{pid_b}")
                    if folder_a == folder_b:
                        continue

                    # ไม่บังคับ cross-platform — โมเดลเรียนจาก same_platform feature เอง

                    key = tuple(sorted([pid_a, pid_b]))
                    if key in existing:
                        continue

                    # คำนวณ similarity
                    ua = _safe_str(ra.get('userName', ''))
                    ub = _safe_str(rb.get('userName', ''))
                    sim_name = _name_similarity(ua, ub)

                    # ถ้าชื่อคล้ายกัน ≥ threshold → hard negative candidate
                    if sim_name >= threshold:
                        candidates.append((key, pid_a, pid_b, sim_name))

        # sort by similarity (คล้ายสุดก่อน = hardest negatives)
        candidates.sort(key=lambda x: x[3], reverse=True)
        candidates = candidates[:n_target]

        pairs = []
        for key, pid_a, pid_b, sim in candidates:
            pairs.append({
                'profile_id_a':  pid_a,
                'profile_id_b':  pid_b,
                'platform_a':    pid_to_plat.get(pid_a, ''),
                'platform_b':    pid_to_plat.get(pid_b, ''),
                'label':         0,
                'pair_type':     'hard_negative',
                'hard_name_sim': round(sim, 4),  # เก็บ score ไว้ debug / ablation
            })

        # cleanup
        if '_name_prefix3' in self.df.columns:
            self.df.drop(columns=['_name_prefix3'], inplace=True)

        df_hard = pd.DataFrame(pairs)
        print(f"[PairBuilder] Hard negatives: {len(df_hard)}"
              f"  (target={n_target}, candidates={len(candidates)})")
        return df_hard

    # ──────────────────────────────────────────────────────────
    # BUILD ALL PAIRS
    # ──────────────────────────────────────────────────────────

    def build(self,
              neg_ratio:      float = NEG_TO_POS_RATIO,
              hard_neg_ratio: float = HARD_NEG_RATIO,
              hard_threshold: float = HARD_NEG_THRESHOLD) -> pd.DataFrame:
        """
        สร้าง labeled_pairs ครบทั้ง 3 ประเภท แล้ว concat + shuffle

        Returns
        -------
        pd.DataFrame : labeled_pairs พร้อม columns:
                       profile_id_a, profile_id_b, platform_a, platform_b,
                       label, pair_type
        """
        print("\n" + "=" * 55)
        print("Stage 8: Pair Construction & Label Building")
        print("=" * 55)

        # (1) Positive
        print("\n[1/3] Generating positive pairs ...")
        df_pos = self.generate_positive_pairs()

        # สร้าง set ของ positive pairs สำหรับ dedup
        pos_set = {
            tuple(sorted([str(r['profile_id_a']), str(r['profile_id_b'])]))
            for _, r in df_pos.iterrows()
        }

        # (2) Random negatives
        print("\n[2/3] Generating random negatives ...")
        df_rand = self.generate_random_negatives(
            n_positive=len(df_pos),
            ratio=neg_ratio,
            positive_pid_set=pos_set,
        )

        rand_set = {
            tuple(sorted([str(r['profile_id_a']), str(r['profile_id_b'])]))
            for _, r in df_rand.iterrows()
        }

        # (3) Hard negatives
        print("\n[3/3] Generating hard negatives ...")
        df_hard = self.generate_hard_negatives(
            n_positive=len(df_pos),
            ratio=hard_neg_ratio,
            threshold=hard_threshold,
            positive_pid_set=pos_set,
            random_neg_set=rand_set,
        )

        # Concat + shuffle
        df_all = pd.concat([df_pos, df_rand, df_hard], ignore_index=True)
        df_all = df_all.sample(frac=1, random_state=self.seed).reset_index(drop=True)

        # report
        print("\n" + "─" * 45)
        print("Label distribution:")
        print(f"  Positive (label=1):  {(df_all['label'] == 1).sum():>7,}")
        print(f"  Negative (label=0):  {(df_all['label'] == 0).sum():>7,}")
        print(f"  Total pairs:         {len(df_all):>7,}")
        print(f"  Pos/Neg ratio:       1 : {(df_all['label'] == 0).sum() / max((df_all['label'] == 1).sum(), 1):.1f}")
        print()
        print("Pair type breakdown:")
        for ptype, cnt in df_all['pair_type'].value_counts().items():
            print(f"  {ptype:<20} {cnt:>7,}")
        print()
        print("Platform pair distribution (positive):")
        pos_only = df_all[df_all['label'] == 1].copy()
        pos_only['plat_pair'] = pos_only.apply(
            lambda r: ' x '.join(sorted([r['platform_a'], r['platform_b']])), axis=1)
        for pp, cnt in pos_only['plat_pair'].value_counts().items():
            print(f"  {pp:<35} {cnt:>6,}")

        return df_all


# ─── quality checks ──────────────────────────────────────────────────────────

def run_quality_checks(labeled_pairs: pd.DataFrame,
                       df_profiles: pd.DataFrame) -> dict:
    """
    ตรวจสอบคุณภาพของ labeled_pairs ก่อน save

    Returns dict ของ QA results
    """
    print("\n" + "=" * 45)
    print("Quality Checks")
    print("=" * 45)

    results = {}

    # 1. ไม่มี duplicate pairs
    pair_keys = labeled_pairs.apply(
        lambda r: tuple(sorted([str(r['profile_id_a']), str(r['profile_id_b'])])),
        axis=1,
    )
    n_dupes = pair_keys.duplicated().sum()
    results['no_duplicate_pairs'] = n_dupes == 0
    print(f"  [{'PASS' if n_dupes == 0 else 'FAIL'}] No duplicate pairs: {n_dupes} duplicates found")

    # 2. ไม่มี self-pairs
    self_pairs = (labeled_pairs['profile_id_a'].astype(str) == labeled_pairs['profile_id_b'].astype(str)).sum()
    results['no_self_pairs'] = self_pairs == 0
    print(f"  [{'PASS' if self_pairs == 0 else 'FAIL'}] No self-pairs: {self_pairs}")

    # 3. ไม่มี positive pair ปนใน negative
    pos_keys = set(pair_keys[labeled_pairs['label'] == 1])
    neg_keys = set(pair_keys[labeled_pairs['label'] == 0])
    overlap  = len(pos_keys & neg_keys)
    results['no_pos_neg_overlap'] = overlap == 0
    print(f"  [{'PASS' if overlap == 0 else 'FAIL'}] No pos/neg overlap: {overlap}")

    # 4. profile_ids ทั้งหมดมีอยู่ใน df_profiles — ใช้ str ทั้งหมด
    valid_pids    = set(df_profiles['profile_id'].astype(str).tolist())
    all_pair_pids = (set(labeled_pairs['profile_id_a'].astype(str)) |
                     set(labeled_pairs['profile_id_b'].astype(str)))
    orphan = all_pair_pids - valid_pids
    results['all_pids_valid'] = len(orphan) == 0
    print(f"  [{'PASS' if len(orphan) == 0 else 'FAIL'}] All profile_ids valid: {len(orphan)} orphan IDs")

    # 5. class imbalance ratio
    n_pos = (labeled_pairs['label'] == 1).sum()
    n_neg = (labeled_pairs['label'] == 0).sum()
    ratio = n_neg / max(n_pos, 1)
    results['class_ratio'] = round(ratio, 2)
    print(f"  [INFO] Class ratio (neg/pos): {ratio:.1f} : 1")

    # 6. hard negatives มี similarity สูงกว่า random negatives ไหม
    # (เป็น sanity check — ถ้าเป็น hard ต้องคล้ายกว่า)
    print(f"  [INFO] Hard negatives generated: "
          f"{(labeled_pairs['pair_type'] == 'hard_negative').sum():,}")

    all_pass = all(v for k, v in results.items() if k != 'class_ratio')
    results['all_pass'] = all_pass
    print(f"\n  {'✅ All checks passed!' if all_pass else '❌ Some checks FAILED'}")

    return results


# ─── main ─────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import pickle

    # ─── 1. Load data ───────────────────────────────────────────
    def _find_csv(name: str) -> Optional[Path]:
        cwd = Path.cwd()
        for base in [cwd] + list(cwd.parents)[:4]:
            p = base / name
            if p.exists():
                return p
        for base in [cwd] + list(cwd.parents)[:2]:
            hits = list(base.rglob(name))
            if hits:
                return hits[0]
        return None

    # hardcode path เครื่องตัวเอง — ถ้าไม่เจอจะค้นหาอัตโนมัติ
    HARDCODE = Path(r"d:\66070260-Year3_Term2\Project1\Code\Project-for-Work\data-for-project\nomalized_profiles.csv")

    CSV_PATH = HARDCODE if HARDCODE.exists() else (
        _find_csv("nomalized_profiles.csv") or _find_csv("combined_profiles.csv")
    )

    if CSV_PATH is None:
        raise FileNotFoundError(
            "ไม่พบ nomalized_profiles.csv หรือ combined_profiles.csv\n"
            "วาง file ไว้ใน directory เดียวกับ script หรือแก้ HARDCODE path ด้านบน"
        )

    df = pd.read_csv(CSV_PATH)
    if 'profile_id' not in df.columns:
        df['profile_id'] = range(len(df))
    print(f"Loaded: {CSV_PATH}  shape={df.shape}")

    # เติม user_folder ที่หายไปด้วย profile_id (สำหรับ single-platform profiles)
    df['user_folder'] = df['user_folder'].fillna(df['profile_id'].astype(str))

    print(f"Profiles: {len(df):,}  |  Entities: {df['user_folder'].nunique():,}")
    print(f"Platforms: {df['platform'].value_counts().to_dict()}")

    # ─── 2. Build pairs ─────────────────────────────────────────
    builder = PairBuilder(df, seed=RANDOM_SEED)

    labeled_pairs = builder.build(
        neg_ratio      = NEG_TO_POS_RATIO,
        hard_neg_ratio = HARD_NEG_RATIO,
        hard_threshold = HARD_NEG_THRESHOLD,
    )

    # ─── 3. Quality checks ──────────────────────────────────────
    qa_results = run_quality_checks(labeled_pairs, df)

    # ─── 4. Save ────────────────────────────────────────────────
    labeled_pairs.to_parquet("labeled_pairs.parquet", index=False)

    stats = {
        "total_pairs":      int(len(labeled_pairs)),
        "positive_pairs":   int((labeled_pairs['label'] == 1).sum()),
        "negative_pairs":   int((labeled_pairs['label'] == 0).sum()),
        "random_negatives": int((labeled_pairs['pair_type'] == 'random_negative').sum()),
        "hard_negatives":   int((labeled_pairs['pair_type'] == 'hard_negative').sum()),
        "class_ratio":      round((labeled_pairs['label'] == 0).sum() /
                                  max((labeled_pairs['label'] == 1).sum(), 1), 2),
        "qa":               qa_results,
        "config": {
            "random_seed":        RANDOM_SEED,
            "neg_to_pos_ratio":   NEG_TO_POS_RATIO,
            "hard_neg_ratio":     HARD_NEG_RATIO,
            "hard_neg_threshold": HARD_NEG_THRESHOLD,
            "cross_platform_only": CROSS_PLATFORM_ONLY,
        }
    }
    with open("pair_stats.json", "w") as f:
        json.dump(stats, f, indent=2)

    print("\n✅ Stage 8 Complete!")
    print(f"   labeled_pairs.parquet  ({len(labeled_pairs):,} pairs)")
    print(f"   pair_stats.json")
    print(f"\n   Positive : {stats['positive_pairs']:,}")
    print(f"   Random-  : {stats['random_negatives']:,}")
    print(f"   Hard-    : {stats['hard_negatives']:,}")
    print(f"   Ratio    : 1 : {stats['class_ratio']}")

Loaded: d:\66070260-Year3_Term2\Project1\Code\Project-for-Work\data-for-project\nomalized_profiles.csv  shape=(36807, 21)
Profiles: 36,807  |  Entities: 14,364
Platforms: {'twitter': 13960, 'googleplus': 11890, 'instagram': 10957}
[PairBuilder] Loaded 36807 profiles, 14364 entities, 3 platforms

Stage 8: Pair Construction & Label Building

[1/3] Generating positive pairs ...


KeyboardInterrupt: 